## Example 1: Basic File Locking with InterProcessLock

In [ ]:
# Example 1: Basic File Locking with InterProcessLock
print("Fasteners - Basic File Locking")

import fasteners
import tempfile
import time
from pathlib import Path

# Create a temporary lock file
with tempfile.TemporaryDirectory() as temp_dir:
    lock_file = Path(temp_dir) / "my_process.lock"
    
    # Create an inter-process lock
    lock = fasteners.InterProcessLock(str(lock_file))
    
    # Acquire the lock
    acquired = lock.acquire(blocking=False)
    print(f"Lock acquired: {acquired}")
    
    if acquired:
        print("Performing critical work...")
        time.sleep(0.1)  # Simulate work
        lock.release()
        print("Lock released")
    
    # Using context manager
    with fasteners.InterProcessLock(str(lock_file)):
        print("Inside locked context")
    
    print("Example 1 completed")

## Example 2: Reader-Writer Locks for Concurrent Access

In [ ]:
# Example 2: Reader-Writer Locks for Concurrent Access
print("Fasteners - Reader-Writer Locks")

import fasteners
import threading
import time

# Create a reader-writer lock
rw_lock = fasteners.ReaderWriterLock()
shared_data = {"counter": 0}

def reader_task(reader_id):
    with rw_lock.read_lock():
        value = shared_data["counter"]
        print(f"Reader {reader_id}: Read counter={value}")
        time.sleep(0.1)

def writer_task(writer_id):
    with rw_lock.write_lock():
        old_value = shared_data["counter"]
        shared_data["counter"] += 1
        print(f"Writer {writer_id}: Updated {old_value} -> {shared_data['counter']}")
        time.sleep(0.05)

# Test concurrent access
threads = []
for i in range(2):
    threads.append(threading.Thread(target=reader_task, args=(i+1,)))
    threads.append(threading.Thread(target=writer_task, args=(i+1,)))

for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Final counter: {shared_data['counter']}")
print("Example 2 completed")

## Example 3: Lock Timeouts and Non-blocking Acquisition

In [ ]:
# Example 3: Lock Timeouts and Non-blocking Acquisition
print("Fasteners - Lock Timeouts")

import fasteners
import tempfile
import time
from pathlib import Path

with tempfile.TemporaryDirectory() as temp_dir:
    lock_file = Path(temp_dir) / "timeout_lock"
    
    lock1 = fasteners.InterProcessLock(str(lock_file))
    lock2 = fasteners.InterProcessLock(str(lock_file))
    
    # First lock acquires successfully
    acquired1 = lock1.acquire(blocking=False)
    print(f"Lock 1 acquired: {acquired1}")
    
    if acquired1:
        # Second lock should timeout
        start_time = time.time()
        acquired2 = lock2.acquire(timeout=0.5)
        duration = time.time() - start_time
        
        print(f"Lock 2 acquired: {acquired2} (after {duration:.3f}s)")
        
        lock1.release()
        print("Lock 1 released")
        
        # Now lock2 should succeed
        acquired2_retry = lock2.acquire(timeout=0.5)
        print(f"Lock 2 retry: {acquired2_retry}")
        
        if acquired2_retry:
            lock2.release()
    
    print("Example 3 completed")

## Example 4: Thread Synchronization with threading.Semaphore

In [ ]:
# Example 4: Thread Synchronization with threading.Semaphore
print("Fasteners - Semaphore Pattern")

import threading
import time

# Create semaphore with max 2 concurrent workers
semaphore = threading.Semaphore(2)
active_workers = {"count": 0}

def worker_process(worker_id):
    acquired = semaphore.acquire(timeout=1.0)
    
    if acquired:
        active_workers["count"] += 1
        current = active_workers["count"]
        
        print(f"Worker {worker_id}: Started (active: {current})")
        time.sleep(0.2)  # Simulate work
        
        active_workers["count"] -= 1
        print(f"Worker {worker_id}: Finished")
        
        semaphore.release()
    else:
        print(f"Worker {worker_id}: Timeout")

# Start 4 workers (only 2 can run concurrently)
workers = []
for i in range(4):
    worker = threading.Thread(target=worker_process, args=(i+1,))
    workers.append(worker)
    worker.start()

for worker in workers:
    worker.join()

print("Example 4 completed")

## Example 5: Advanced Lock Patterns with Context Managers

In [ ]:
# Example 5: Advanced Lock Patterns with Context Managers
print("Fasteners - Advanced Lock Patterns")

import fasteners
import tempfile
import threading
import time
from pathlib import Path

class SafeCounter:
    def __init__(self, lock_path):
        self.lock = threading.Lock()  # Use threading.Lock for thread safety
        self.file_lock = fasteners.InterProcessLock(str(lock_path))
        self.value = 0
    
    def increment(self, worker_id):
        with self.lock:  # Thread-safe increment
            old_value = self.value
            time.sleep(0.01)  # Simulate processing
            self.value += 1
            print(f"Worker {worker_id}: {old_value} -> {self.value}")
            return self.value
    
    def save_to_file(self):
        with self.file_lock:  # Process-safe file operations
            print(f"Saving counter value {self.value} to file")

# Test thread-safe counter
with tempfile.TemporaryDirectory() as temp_dir:
    lock_file = Path(temp_dir) / "counter.lock"
    counter = SafeCounter(lock_file)
    
    threads = []
    for i in range(3):
        thread = threading.Thread(target=counter.increment, args=(i+1,))
        threads.append(thread)
        thread.start()
    
    for thread in threads:
        thread.join()
    
    counter.save_to_file()
    print(f"Final value: {counter.value}")
    print("Example 5 completed")